# 📁 Day 01a: File Organization — Allocation Methods & Inodes

---

## 🎯 What You'll Master Today
- Contiguous, linked, and indexed file allocation
- FAT (File Allocation Table)
- Unix inode structure
- Trade-offs: sequential vs random access

---

```
╔══════════════════════════════════════════════════════════════╗
║           FILE ALLOCATION METHODS                            ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Contiguous: [B0][B1][B2][B3] — fast but fragmentation      ║
║              start=B0, length=4                              ║
║                                                              ║
║  Linked:     [B0]→[B5]→[B2]→[B8] — no frag, slow random    ║
║              each block has pointer to next                  ║
║                                                              ║
║  Indexed:    Index Block: [0→B4, 1→B7, 2→B2, 3→B9]         ║
║              direct random access, wastes index space        ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. File Allocation Strategies Simulation
# ============================================

class ContiguousAllocation:
    """File stored in consecutive blocks."""
    def __init__(self, disk_size: int):
        self.disk = [None] * disk_size
    
    def allocate(self, filename: str, size: int) -> bool:
        # Find first contiguous free region
        count = 0
        start = -1
        for i in range(len(self.disk)):
            if self.disk[i] is None:
                if count == 0:
                    start = i
                count += 1
                if count == size:
                    for j in range(start, start + size):
                        self.disk[j] = filename
                    print(f"  Contiguous: '{filename}' at blocks {start}-{start+size-1}")
                    return True
            else:
                count = 0
        print(f"  Contiguous: '{filename}' FAILED — no {size} contiguous blocks!")
        return False


class LinkedAllocation:
    """File stored as linked list of blocks."""
    def __init__(self, disk_size: int):
        self.disk = [None] * disk_size
        self.directory = {}  # filename → first block
    
    def allocate(self, filename: str, size: int) -> bool:
        free_blocks = [i for i in range(len(self.disk)) if self.disk[i] is None]
        if len(free_blocks) < size:
            return False
        
        blocks = free_blocks[:size]
        self.directory[filename] = blocks[0]
        for i, blk in enumerate(blocks):
            next_blk = blocks[i+1] if i+1 < len(blocks) else -1
            self.disk[blk] = (filename, next_blk)
        
        chain = " → ".join(str(b) for b in blocks)
        print(f"  Linked: '{filename}' chain: {chain}")
        return True
    
    def read_sequential(self, filename: str) -> list[int]:
        blocks = []
        current = self.directory.get(filename)
        while current is not None and current != -1:
            blocks.append(current)
            _, current = self.disk[current]
        return blocks


class IndexedAllocation:
    """File has an index block listing all data blocks."""
    def __init__(self, disk_size: int):
        self.disk = [None] * disk_size
        self.directory = {}  # filename → index block number
    
    def allocate(self, filename: str, size: int) -> bool:
        free_blocks = [i for i in range(len(self.disk)) if self.disk[i] is None]
        if len(free_blocks) < size + 1:  # +1 for index block
            return False
        
        index_block = free_blocks[0]
        data_blocks = free_blocks[1:size+1]
        
        self.disk[index_block] = data_blocks  # index block stores pointers
        for blk in data_blocks:
            self.disk[blk] = filename
        self.directory[filename] = index_block
        
        print(f"  Indexed: '{filename}' index@{index_block}, data@{data_blocks}")
        print(f"    Random access block 2: directly → block {data_blocks[2] if len(data_blocks) > 2 else 'N/A'}")
        return True


# Demo all three
print("  === Contiguous ===")
ca = ContiguousAllocation(20)
ca.allocate("file_a", 3)
ca.allocate("file_b", 5)

print("\n  === Linked ===")
la = LinkedAllocation(20)
la.allocate("file_a", 3)
la.allocate("file_b", 5)
print(f"  Sequential read of file_a: {la.read_sequential('file_a')}")

print("\n  === Indexed ===")
ia = IndexedAllocation(20)
ia.allocate("file_a", 3)
ia.allocate("file_b", 5)

In [ ]:
# ============================================
# 2. Unix Inode Structure Simulation
# ============================================

class Inode:
    """
    Simplified Unix inode with:
    - 12 direct pointers (each → 4KB block)
    - 1 single indirect (→ block of 1024 pointers)
    - 1 double indirect (→ block of pointers to pointer blocks)
    """
    BLOCK_SIZE = 4096       # 4KB
    PTRS_PER_BLOCK = 1024   # 4KB / 4 bytes per pointer
    
    def __init__(self, filename: str):
        self.filename = filename
        self.direct = [None] * 12       # 12 direct block pointers
        self.single_indirect = None     # → block of pointers
        self.double_indirect = None     # → block of pointer blocks
    
    def max_file_size(self) -> int:
        direct = 12 * self.BLOCK_SIZE
        single = self.PTRS_PER_BLOCK * self.BLOCK_SIZE
        double = self.PTRS_PER_BLOCK ** 2 * self.BLOCK_SIZE
        return direct + single + double
    
    def explain(self):
        bs = self.BLOCK_SIZE
        ppb = self.PTRS_PER_BLOCK
        
        direct_bytes = 12 * bs
        single_bytes = ppb * bs
        double_bytes = ppb * ppb * bs
        total = direct_bytes + single_bytes + double_bytes
        
        print(f"  Unix Inode Structure ({self.filename})")
        print(f"  {'═' * 50}")
        print(f"  Block Size:        {bs // 1024}KB")
        print(f"  Pointers/Block:    {ppb}")
        print(f"  ────────────────────────────────")
        print(f"  12 Direct:         12 × {bs//1024}KB = {direct_bytes // 1024}KB")
        print(f"  1 Single Indirect: {ppb} × {bs//1024}KB = {single_bytes // (1024*1024)}MB")
        print(f"  1 Double Indirect: {ppb}² × {bs//1024}KB = {double_bytes // (1024*1024*1024)}GB")
        print(f"  ────────────────────────────────")
        print(f"  Max File Size:     ~{total // (1024*1024*1024)}GB")
        print(f"\n  💡 Small files (<48KB) use ONLY direct pointers — fast!")
        print(f"  💡 Most Unix files are small → inode design is optimized")


inode = Inode("example.txt")
inode.explain()

```
╔════════════════╦════════════╦════════════╦═══════════════╗
║  Method        ║ Sequential ║  Random    ║ Fragmentation ║
╠════════════════╬════════════╬════════════╬═══════════════╣
║  Contiguous    ║  Fast ✅   ║  Fast ✅   ║  External ❌  ║
║  Linked        ║  OK        ║  Slow ❌   ║  None ✅      ║
║  Indexed       ║  OK        ║  Fast ✅   ║  Index waste  ║
║  Unix Inode    ║  Fast      ║  Fast      ║  Balanced ✅  ║
╚════════════════╩════════════╩════════════╩═══════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Contiguous vs linked allocation? | Contiguous: fast access, external frag. Linked: no frag, terrible random access |
| 2 | What is an inode? | Metadata structure: permissions, size, timestamps, 12 direct + indirect block pointers |
| 3 | Max file size with inode? | ~4GB with 4KB blocks (12 direct + single + double indirect) |
| 4 | What is FAT? | Global linked list table. FAT[i] = next block. Used by FAT32, simpler than inode |
| 5 | Why are most files small? | Studies show >90% files < 1KB. Inode direct pointers optimized for this |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Contiguous: fastest but suffers external fragmentation│
## │  • Linked: no fragmentation but O(n) random access       │
## │  • Indexed/Inode: best of both, used by Unix/Linux       │
## │  • FAT: simpler alternative, used by USB/SD cards        │
## │  • Inode optimized for small files (direct pointers)     │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Disk Scheduling** — FCFS, SSTF, SCAN, C-SCAN